# Baseline Evaluation

## Objective

Before fine-tuning the model, we evaluate the base Qwen model on the held-out test dataset.

The purpose of baseline evaluation is to establish the model's initial
tool-calling capability and provide a reference point for measuring
improvements after fine-tuning.

The evaluation measures whether the model can:

1. Select the correct tool.
2. Generate the correct number of tool calls.
3. Extract the correct arguments from the user query.
4. Generate multiple tool calls when required.
5. Maintain the correct sequence of tool calls.

The same test dataset will later be used to evaluate the fine-tuned model

In [ ]:
import json
import re
import time
import gc

import torch
import pandas as pd
from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ================================================================
# LOAD BASE QWEN MODEL AND TOKENIZER
# ================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

print("LOADING BASE QWEN MODEL")


# ------------------------------------------------
# Load tokenizer
# ------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# ------------------------------------------------
# Load model
# ------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# ------------------------------------------------
# Evaluation mode: Set model to inference mode
# ------------------------------------------------
model.eval()

# ------------------------------------------------
# Verify
# ------------------------------------------------
print("\nModel loaded successfully!")

print("\nModel:")
print(MODEL_NAME)

print("\nTokenizer type:")
print(type(tokenizer))

print("\nModel device:")
print(model.device)

print("\nGPU available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

print("\nModel is ready for baseline inference.")

## Prompt Construction for Tool Calling

Qwen's tool-calling capability relies on a specific chat template that formats the
user query, available tools, and generation context into a structured prompt.

For each example, the following information is provided:

- **User Query:** The task that needs to be completed.
- **Available Tools:** A set of functions that the model can select from.
- **Tool Schema:** Each tool contains its name, description, and parameter definitions.

The tokenizer's `apply_chat_template()` method is used to construct the prompt
according to Qwen's expected tool-calling protocol.

The model is instructed to generate tool calls using the following structure:

<tool_call>
{"name": "<tool_name>", "arguments": {...}}
</tool_call>

The generated output is later parsed and compared with the expected tool calls
provided in the dataset.

In [ ]:
# ================================================================
# PROMPT CONSTRUCTION
# ================================================================

import json


def create_qwen_prompt(example):
    """
    Constructs a Qwen-compatible tool-calling prompt.

    The prompt contains:
        1. User query
        2. Available tool definitions
        3. Qwen's tool-calling protocol

    Parameters
    ----------
    example : dict
        A dataset example containing:
        - query
        - tools

    Returns
    -------
    str
        Formatted prompt ready for tokenization and inference.
    """

    # ------------------------------------------------------------
    # Extract query and tools
    # ------------------------------------------------------------

    query = example["query"]

    tools = json.loads(example["tools"])

    # ------------------------------------------------------------
    # Create chat messages
    # ------------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": query
        }
    ]

    # ------------------------------------------------------------
    # Apply Qwen chat template
    # ------------------------------------------------------------

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

In [ ]:
# ================================================================
# INSPECT PROMPT CONSTRUCTION ON ONE EXAMPLE
# ================================================================

example = test_dataset[0]

formatted_prompt = create_qwen_prompt(example)


print("USER QUERY")

print(example["query"])

print("AVAILABLE TOOLS")
print(json.dumps(
    json.loads(example["tools"]),
    indent=2
))

print("FORMATTED QWEN PROMPT")
print(formatted_prompt)

## Base Model Inference on One Example

In this step, we evaluate the base Qwen model on a single tool-calling example.

The inference pipeline consists of:

1. Constructing the Qwen-compatible prompt.
2. Tokenizing the formatted prompt.
3. Moving the tokenized inputs to the model device.
4. Generating the model response.
5. Separating the generated tokens from the original input.
6. Decoding the generated output.


In [ ]:
# ================================================================
# BASE MODEL INFERENCE ON ONE EXAMPLE
# ================================================================

import torch


# ------------------------------------------------
# 1. Construct Qwen-formatted prompt
# ------------------------------------------------

formatted_prompt = create_qwen_prompt(example)


# ------------------------------------------------
# 2. Tokenize the prompt
# ------------------------------------------------

model_inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
)


# ------------------------------------------------
# 3. Move inputs to the model device
# ------------------------------------------------

model_inputs = {
    key: value.to(model.device)
    for key, value in model_inputs.items()
}


# ------------------------------------------------
# Display tokenization information
# ------------------------------------------------


print("TOKENIZATION INFORMATION")
print(f"Input token length: {model_inputs['input_ids'].shape[1]}")


# ------------------------------------------------
# 4. Generate model response
# ------------------------------------------------

print("\nGenerating response...")

with torch.inference_mode():

    output_ids = model.generate(

        **model_inputs,

        max_new_tokens=256,

        do_sample=False,

        pad_token_id=tokenizer.eos_token_id

    )


# ------------------------------------------------
# 5. Separate generated tokens from input tokens
# ------------------------------------------------

input_length = model_inputs["input_ids"].shape[1]

generated_ids = output_ids[:, input_length:]


# ------------------------------------------------
# 6. Decode generated output
# ------------------------------------------------

raw_response = tokenizer.decode(

    generated_ids[0],

    skip_special_tokens=False

)


# ------------------------------------------------
# Display results
# ------------------------------------------------

print("MODEL GENERATION RESULTS")
print(f"\nInput length: {input_length} tokens")
print(f"Generated token length: {generated_ids.shape[1]} tokens")

print("RAW MODEL RESPONSE")
print(raw_response)

## Parsing Generated Tool Calls

The base model generates tool calls as text using Qwen's tool-calling format:

<tool_call>
{"name": "<tool_name>", "arguments": {...}}
</tool_call>

Before evaluating the model's predictions, the generated response must be parsed
into structured Python objects.

The parser extracts each `<tool_call>` block and converts the JSON content into
a dictionary containing:

- Tool name
- Tool arguments

This enables direct comparison between the model's predicted tool calls and the
expected tool calls provided in the dataset.

In [ ]:
# ================================================================
# TOOL CALL PARSING
# ================================================================

import re
import json


def parse_tool_calls(raw_response):
    """
    Extract tool calls from the raw model response.

    Parameters
    ----------
    raw_response : str
        Raw text generated by the model.

    Returns
    -------
    list
        List of parsed tool call dictionaries.
    """

    # ------------------------------------------------------------
    # Find all content between <tool_call> tags
    # ------------------------------------------------------------

    tool_call_blocks = re.findall(
        r"<tool_call>\s*(.*?)\s*</tool_call>",
        raw_response,
        re.DOTALL
    )

    parsed_calls = []

    # ------------------------------------------------------------
    # Parse each JSON tool call
    # ------------------------------------------------------------

    for block in tool_call_blocks:

        try:
            tool_call = json.loads(block)
            parsed_calls.append(tool_call)

        except json.JSONDecodeError:
            # Ignore malformed tool calls
            continue

    return parsed_calls

In [ ]:
# ================================================================
# PARSE THE MODEL RESPONSE
# ================================================================

predicted_calls = parse_tool_calls(raw_response)

print("=" * 80)
print("PARSED TOOL CALLS")
print("=" * 80)

print(json.dumps(predicted_calls, indent=2))

## Evaluation of a Single Example

After parsing the model's generated tool calls, the predicted calls are compared
with the expected tool calls provided in the dataset.

Three basic evaluation criteria are used:

1. **Exact Match**  
   The predicted sequence of tool calls exactly matches the expected sequence,
   including tool names, arguments, and call order.

2. **Tool Name Match**  
   The predicted tool names match the expected tool names.

3. **Arguments Match**  
   The arguments associated with the predicted tool calls exactly match the
   expected arguments.

This single-example evaluation is used to verify the evaluation pipeline before
scaling the experiment to larger subsets of the test dataset.

In [ ]:
# ================================================================
# EVALUATE ONE EXAMPLE
# ================================================================

# Load expected tool calls from the dataset
expected_calls = json.loads(example["answers"])


# ------------------------------------------------
# Exact Match
# ------------------------------------------------

exact_match = predicted_calls == expected_calls


# ------------------------------------------------
# Tool Name Match
# ------------------------------------------------

expected_tool_names = [
    call["name"]
    for call in expected_calls
]

predicted_tool_names = [
    call["name"]
    for call in predicted_calls
]

tool_name_match = (
    expected_tool_names == predicted_tool_names
)


# ------------------------------------------------
# Arguments Match
# ------------------------------------------------

expected_arguments = [
    call.get("arguments", {})
    for call in expected_calls
]

predicted_arguments = [
    call.get("arguments", {})
    for call in predicted_calls
]

arguments_match = (
    expected_arguments == predicted_arguments
)


# ------------------------------------------------
# Display Results
# ------------------------------------------------

print("=" * 80)
print("SINGLE EXAMPLE EVALUATION")
print("=" * 80)


print("\nEXPECTED TOOL CALLS:")
print(json.dumps(expected_calls, indent=2))


print("\nPREDICTED TOOL CALLS:")
print(json.dumps(predicted_calls, indent=2))


print("\n" + "=" * 80)
print("EVALUATION METRICS")
print("=" * 80)

print(f"Exact Match:      {exact_match}")
print(f"Tool Name Match:  {tool_name_match}")
print(f"Arguments Match:  {arguments_match}")

print(f"\nExpected Call Count:  {len(expected_calls)}")
print(f"Predicted Call Count: {len(predicted_calls)}")

## Small-Scale Baseline Evaluation

Before evaluating the base model on the complete test dataset, a smaller,
stratified subset of 40 examples was used to validate the inference and
evaluation pipeline.

The test examples were grouped according to the number of expected tool calls:

- `1_call`
- `2_calls`
- `3_calls`
- `4_plus_calls`

Ten examples were sampled from each complexity group, resulting in a balanced
evaluation subset of 40 examples.

This stratified sampling allows the initial baseline experiment to evaluate
model performance across different levels of tool-calling complexity while
keeping computational requirements manageable.

In [ ]:
# ================================================================
# CREATE STRATIFIED 40-SAMPLE BASELINE SUBSET
# ================================================================

import random

# Ensure reproducibility
random.seed(42)


# Number of samples from each complexity group
samples_per_group = 10


# Store sampled examples
baseline_subset = []


# Complexity groups
complexity_groups = [
    "1_call",
    "2_calls",
    "3_calls",
    "4_plus_calls"
]


# ------------------------------------------------
# Sample 10 examples from each complexity group
# ------------------------------------------------

for group in complexity_groups:

    group_examples = [
        example
        for example in test_dataset
        if example["complexity_group"] == group
    ]

    sampled_examples = random.sample(
        group_examples,
        samples_per_group
    )

    baseline_subset.extend(sampled_examples)


# ------------------------------------------------
# Display sampling results
# ------------------------------------------------

print("=" * 80)
print("BASELINE EVALUATION SUBSET")
print("=" * 80)

print(f"\nTotal samples: {len(baseline_subset)}")

print("\nSamples by complexity group:")

for group in complexity_groups:

    count = sum(
        example["complexity_group"] == group
        for example in baseline_subset
    )

    print(f"{group}: {count}")

## Reusable Tool Call Prediction Pipeline

To evaluate multiple examples efficiently, the individual inference steps are
combined into a reusable prediction function.

For each example, the function:

1. Extracts the user query and available tools.
2. Constructs a Qwen-compatible tool-calling prompt.
3. Tokenizes the prompt.
4. Generates a response using the base model.
5. Extracts only the newly generated tokens.
6. Decodes the generated response.
7. Parses the generated `<tool_call>` blocks.

The function returns both the structured predicted tool calls and the raw model
response for later evaluation and error analysis.

In [ ]:
# ================================================================
# TOOL CALL PREDICTION FUNCTION
# ================================================================

def predict_tool_calls(example, max_new_tokens=256):
    """
    Generate and parse tool calls for a single dataset example.

    Parameters
    ----------
    example : dict
        Dataset example containing query and tool definitions.

    max_new_tokens : int
        Maximum number of tokens generated by the model.

    Returns
    -------
    predicted_calls : list
        Parsed tool calls generated by the model.

    raw_response : str
        Raw generated model response.
    """

    # ------------------------------------------------------------
    # 1. Construct prompt
    # ------------------------------------------------------------

    formatted_prompt = create_qwen_prompt(example)


    # ------------------------------------------------------------
    # 2. Tokenize prompt
    # ------------------------------------------------------------

    model_inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    # Move inputs to model device
    model_inputs = {
        key: value.to(model.device)
        for key, value in model_inputs.items()
    }


    # ------------------------------------------------------------
    # 3. Generate response
    # ------------------------------------------------------------

    with torch.inference_mode():

        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )


    # ------------------------------------------------------------
    # 4. Extract generated tokens
    # ------------------------------------------------------------

    input_length = model_inputs["input_ids"].shape[1]

    generated_ids = output_ids[:, input_length:]


    # ------------------------------------------------------------
    # 5. Decode response
    # ------------------------------------------------------------

    raw_response = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=False
    )


    # ------------------------------------------------------------
    # 6. Parse tool calls
    # ------------------------------------------------------------

    predicted_calls = parse_tool_calls(raw_response)


    return predicted_calls, raw_response

## Baseline Evaluation on 40 Stratified Samples

The complete inference pipeline is now evaluated on the 40-sample stratified
subset.

For each example:

1. A Qwen-compatible prompt is constructed.
2. The base model generates a tool-calling response.
3. Generated tool calls are parsed.
4. Predictions and expected tool calls are stored.

The results from this experiment will be used to calculate baseline evaluation
metrics and analyze performance across different tool-calling complexity levels.

In [ ]:
# ================================================================
# RUN BASELINE EVALUATION ON 40 SAMPLES
# ================================================================

import time
import gc
import torch
import json


# Store results from all predictions
baseline_results = []


print("=" * 80)
print("STARTING BASELINE EVALUATION")
print("=" * 80)

print(f"\nTotal samples: {len(baseline_subset)}")


# ------------------------------------------------
# Run inference on each example
# ------------------------------------------------

start_time = time.time()


for index, example in enumerate(baseline_subset):

    print(f"\nRunning example {index + 1}/{len(baseline_subset)}...")

    try:

        # ----------------------------------------
        # Generate prediction
        # ----------------------------------------

        predicted_calls, raw_response = predict_tool_calls(example)


        # ----------------------------------------
        # Load expected calls
        # ----------------------------------------

        expected_calls = json.loads(example["answers"])


        # ----------------------------------------
        # Store result
        # ----------------------------------------

        baseline_results.append({

            "id": example["id"],

            "query": example["query"],

            "complexity_group": example["complexity_group"],

            "expected_calls": expected_calls,

            "predicted_calls": predicted_calls,

            "raw_response": raw_response

        })


        print("✓ Completed")


    except Exception as e:

        print(f"✗ Error: {e}")

        # Store failed example
        baseline_results.append({

            "id": example["id"],

            "query": example["query"],

            "complexity_group": example["complexity_group"],

            "expected_calls": json.loads(example["answers"]),

            "predicted_calls": [],

            "raw_response": "",

            "error": str(e)

        })


    # ------------------------------------------------
    # Memory cleanup
    # ------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------
# Calculate execution time
# ------------------------------------------------

end_time = time.time()

total_time = end_time - start_time


# ------------------------------------------------
# Display summary
# ------------------------------------------------

print("\n" + "=" * 80)
print("BASELINE EVALUATION COMPLETE")
print("=" * 80)

print(f"\nSamples processed: {len(baseline_results)}")

print(f"Total execution time: {total_time:.2f} seconds")

print(
    f"Average time per sample: "
    f"{total_time / len(baseline_results):.2f} seconds"
)

## Evaluation Metrics

The baseline model is evaluated using multiple metrics to capture different
aspects of tool-calling performance.

### 1. Exact Match Accuracy

A prediction is considered correct only when the complete sequence of predicted
tool calls exactly matches the expected sequence, including:

- Tool names
- Argument names
- Argument values
- Number of tool calls
- Order of tool calls

### 2. Tool Name Accuracy

This metric evaluates whether the model selected the correct sequence of tools,
independent of whether the arguments were correct.

### 3. Arguments Accuracy

This metric evaluates whether the arguments in the predicted tool calls exactly
match the expected arguments.

Using multiple metrics allows us to distinguish between failures caused by
incorrect tool selection and failures caused by incorrect argument generation.

In [ ]:
# ================================================================
# EVALUATION FUNCTION
# ================================================================

def evaluate_prediction(expected_calls, predicted_calls):
    """
    Evaluate predicted tool calls against expected tool calls.
    
    Returns a dictionary containing evaluation metrics.
    """

    # ------------------------------------------------------------
    # Exact Match
    # ------------------------------------------------------------

    exact_match = expected_calls == predicted_calls


    # ------------------------------------------------------------
    # Extract Tool Names
    # ------------------------------------------------------------

    expected_tool_names = [
        call["name"]
        for call in expected_calls
    ]

    predicted_tool_names = [
        call["name"]
        for call in predicted_calls
    ]


    # ------------------------------------------------------------
    # Tool Name Match
    # ------------------------------------------------------------

    tool_name_match = (
        expected_tool_names == predicted_tool_names
    )


    # ------------------------------------------------------------
    # Extract Arguments
    # ------------------------------------------------------------

    expected_arguments = [
        call.get("arguments", {})
        for call in expected_calls
    ]

    predicted_arguments = [
        call.get("arguments", {})
        for call in predicted_calls
    ]


    # ------------------------------------------------------------
    # Arguments Match
    # ------------------------------------------------------------

    arguments_match = (
        expected_arguments == predicted_arguments
    )


    # ------------------------------------------------------------
    # Call Count Match
    # ------------------------------------------------------------

    call_count_match = (
        len(expected_calls) == len(predicted_calls)
    )


    return {
        "exact_match": exact_match,
        "tool_name_match": tool_name_match,
        "arguments_match": arguments_match,
        "call_count_match": call_count_match,
        "expected_call_count": len(expected_calls),
        "predicted_call_count": len(predicted_calls)
    }

In [ ]:
# ================================================================
# APPLY EVALUATION METRICS TO BASELINE RESULTS
# ================================================================

evaluated_results = []


for result in baseline_results:

    # Get evaluation metrics
    metrics = evaluate_prediction(
        expected_calls=result["expected_calls"],
        predicted_calls=result["predicted_calls"]
    )


    # Combine original result with metrics
    evaluated_result = {
        **result,
        **metrics
    }


    evaluated_results.append(evaluated_result)


print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(f"\nTotal evaluated examples: {len(evaluated_results)}")

In [ ]:
# ================================================================
# OVERALL BASELINE PERFORMANCE
# ================================================================

total_examples = len(evaluated_results)


# ------------------------------------------------
# Calculate Accuracy
# ------------------------------------------------

exact_match_accuracy = (
    sum(result["exact_match"] for result in evaluated_results)
    / total_examples
) * 100


tool_name_accuracy = (
    sum(result["tool_name_match"] for result in evaluated_results)
    / total_examples
) * 100


arguments_accuracy = (
    sum(result["arguments_match"] for result in evaluated_results)
    / total_examples
) * 100


call_count_accuracy = (
    sum(result["call_count_match"] for result in evaluated_results)
    / total_examples
) * 100


# ------------------------------------------------
# Display Results
# ------------------------------------------------

print("=" * 80)
print("OVERALL BASELINE PERFORMANCE")
print("=" * 80)

print(f"\nTotal Examples: {total_examples}")

print(
    f"Exact Match Accuracy: "
    f"{exact_match_accuracy:.2f}%"
)

print(
    f"Tool Name Accuracy: "
    f"{tool_name_accuracy:.2f}%"
)

print(
    f"Arguments Accuracy: "
    f"{arguments_accuracy:.2f}%"
)

print(
    f"Call Count Accuracy: "
    f"{call_count_accuracy:.2f}%"
)

## Baseline Performance by Tool-Calling Complexity

To understand how tool-calling complexity affects model performance, the baseline
results are evaluated separately for each complexity group:

- `1_call`
- `2_calls`
- `3_calls`
- `4_plus_calls`

This analysis helps determine whether model performance decreases as the number
of required tool calls increases.

In [ ]:
# ================================================================
# BASELINE PERFORMANCE BY COMPLEXITY
# ================================================================

import pandas as pd


# ------------------------------------------------
# Convert results into a DataFrame
# ------------------------------------------------

results_df = pd.DataFrame(evaluated_results)


# ------------------------------------------------
# Group by complexity and calculate metrics
# ------------------------------------------------

complexity_results = (
    results_df
    .groupby("complexity_group")
    .agg(
        samples=("id", "count"),

        exact_match_accuracy=(
            "exact_match",
            lambda x: x.mean() * 100
        ),

        tool_name_accuracy=(
            "tool_name_match",
            lambda x: x.mean() * 100
        ),

        arguments_accuracy=(
            "arguments_match",
            lambda x: x.mean() * 100
        ),

        call_count_accuracy=(
            "call_count_match",
            lambda x: x.mean() * 100
        )
    )
)


# ------------------------------------------------
# Order complexity groups logically
# ------------------------------------------------

complexity_order = [
    "1_call",
    "2_calls",
    "3_calls",
    "4_plus_calls"
]

complexity_results = complexity_results.reindex(
    complexity_order
)


# ------------------------------------------------
# Display results
# ------------------------------------------------

print("=" * 80)
print("BASELINE PERFORMANCE BY COMPLEXITY")
print("=" * 80)

print()

display(complexity_results.round(2))